# Download Images

In [2]:
import os

zip_url = "https://storage.googleapis.com/bean-flowering/images.zip"
zip_filename = "images.zip"
images_dir = "images"

# Download the zip file
print(f"Downloading {zip_filename} from {zip_url}...")
os.system(f"wget -O {zip_filename} {zip_url}")

# Unzip the file
print(f"Unzipping {zip_filename}...")
os.system(f"unzip -q {zip_filename}")

# Remove the zip file after extraction
os.system(f"rm {zip_filename}")

print(f"Images downloaded and extracted to ./{images_dir}/")

Unzipping images.zip...
Images downloaded and extracted to ./images/


# Part 1: Config & Imports



Set project paths, training parameters, category mapping, and reusable helpers for the YOLO workflow.

In [6]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.2 MB/s eta 0:00:00


In [7]:
import ast
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()

TRAIN_CSV = PROJECT_DIR / "Train.csv"

TEST_CSV = PROJECT_DIR / "Test.csv"

IMAGES_DIR = PROJECT_DIR / "images"


YOLO_DATASET_DIR = PROJECT_DIR / "yolo_dataset"

YOLO_IMAGES_TRAIN = YOLO_DATASET_DIR / "images" / "train"

YOLO_IMAGES_VAL = YOLO_DATASET_DIR / "images" / "val"

YOLO_LABELS_TRAIN = YOLO_DATASET_DIR / "labels" / "train"

YOLO_LABELS_VAL = YOLO_DATASET_DIR / "labels" / "val"

DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"


RUNS_DIR = PROJECT_DIR / "runs"

SUBMISSION_PATH = PROJECT_DIR / "submission.csv"


MODEL_WEIGHTS = (
    "yolo11n-seg.pt"
)

EPOCHS = 10

IMAGE_SIZE = 640

BATCH_SIZE = 8

VAL_SIZE = 0.2

RANDOM_STATE = 42

CONF_THRESHOLD = 0.25

IOU_THRESHOLD = 0.5

N_JOBS_DATA_PREP = 4


CLASS_NAMES = ["Plant_bean", "Flower_open", "Flower_closed"]

CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

TRAIN_REQUIRED_COLS = {"Image_ID", "image_width", "image_height", "bbox", "points", "class"}
TEST_REQUIRED_COLS = {"Image_ID", "image_width", "image_height"}


for path in [TRAIN_CSV, TEST_CSV, IMAGES_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")


print("Config ready")

print(f"Project dir: {PROJECT_DIR}")

print(f"Train CSV:   {TRAIN_CSV}")

print(f"Test CSV:    {TEST_CSV}")

print(f"Images dir:  {IMAGES_DIR}")

print(f"Model:       {MODEL_WEIGHTS}")

print(f"Epochs:      {EPOCHS}")

print(f"Image size:  {IMAGE_SIZE}")

print(f"Batch size:  {BATCH_SIZE}")

print(f"Data prep workers: {N_JOBS_DATA_PREP}")

print(f"Expected Train.csv columns: {sorted(TRAIN_REQUIRED_COLS)}")

print(f"Expected Test.csv columns:  {sorted(TEST_REQUIRED_COLS)}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Config ready
Project dir: /content
Train CSV:   /content/Train.csv
Test CSV:    /content/Test.csv
Images dir:  /content/images
Model:       yolo11n-seg.pt
Epochs:      10
Image size:  640
Batch size:  8
Data prep workers: 4
Expected Train.csv columns: ['Image_ID', 'bbox', 'class', 'image_height', 'image_width', 'points']
Expected Test.csv columns:  ['Image_ID', 'image_height', 'image_width']


# Part 2: Data Preparation

Create YOLO train/validation folders, convert `Train.csv` bounding boxes into YOLO label files, and write the dataset YAML file.

Expected input format:
- `Train.csv`: `Image_ID`, `image_width`, `image_height`, `bbox`, `points`, `class`
- `Test.csv`: `Image_ID`, `image_width`, `image_height`

In [8]:
train_df = pd.read_csv(TRAIN_CSV)

test_df = pd.read_csv(TEST_CSV)


missing_train_cols = TRAIN_REQUIRED_COLS.difference(train_df.columns)

if missing_train_cols:
    raise ValueError(
        f"Train.csv is missing required columns: {sorted(missing_train_cols)}"
    )


missing_test_cols = TEST_REQUIRED_COLS.difference(test_df.columns)

if missing_test_cols:
    raise ValueError(
        f"Test.csv is missing required columns: {sorted(missing_test_cols)}"
    )


invalid_classes = sorted(set(train_df["class"].dropna()) - set(CLASS_NAMES))

if invalid_classes:
    raise ValueError(f"Train.csv contains unsupported classes: {invalid_classes}")


if test_df["Image_ID"].duplicated().any():
    duplicated_count = int(test_df["Image_ID"].duplicated().sum())
    raise ValueError(
        f"Test.csv should contain one row per image. Duplicated Image_ID count: {duplicated_count}"
    )


for folder in [YOLO_IMAGES_TRAIN, YOLO_IMAGES_VAL, YOLO_LABELS_TRAIN, YOLO_LABELS_VAL]:
    folder.mkdir(parents=True, exist_ok=True)


def parse_bbox(value):

    if isinstance(value, str):
        parts = [piece.strip() for piece in value.strip().strip("[]").split(",")]

        if len(parts) != 4:
            raise ValueError(f"Invalid bbox string: {value}")

        x, y, w, h = [float(piece) for piece in parts]

        return x, y, w, h

    if isinstance(value, (list, tuple)) and len(value) == 4:
        return [float(v) for v in value]

    raise ValueError(f"Unsupported bbox value: {value}")


def parse_points(value):
    if pd.isna(value):
        return []

    if isinstance(value, (list, tuple)):
        pts = []
        for p in value:
            if isinstance(p, (list, tuple)) and len(p) >= 2:
                pts.append((float(p[0]), float(p[1])))
        return pts

    s = str(value).strip()
    if not s:
        return []

    # Try Python-literal style first: [[x,y], [x,y], ...]
    if s[0] in "[(":
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                pts = []
                for p in parsed:
                    if isinstance(p, (list, tuple)) and len(p) >= 2:
                        pts.append((float(p[0]), float(p[1])))
                return pts
        except Exception:
            pass

    # Fallback: x,y;x,y;...
    pts = []
    for token in s.split(";"):
        token = token.strip()
        if not token:
            continue
        pair = [v.strip() for v in token.split(",")]
        if len(pair) < 2:
            continue
        try:
            pts.append((float(pair[0]), float(pair[1])))
        except ValueError:
            continue

    return pts


def normalize_polygon(points, image_width, image_height):
    norm = []
    for x, y in points:
        xn = min(1.0, max(0.0, float(x) / image_width))
        yn = min(1.0, max(0.0, float(y) / image_height))
        norm.append((xn, yn))
    return norm


def bbox_to_polygon(x, y, w, h):
    return [
        (x, y),
        (x + w, y),
        (x + w, y + h),
        (x, y + h),
    ]


unique_ids = train_df["Image_ID"].drop_duplicates().tolist()

train_ids, val_ids = train_test_split(
    unique_ids, test_size=VAL_SIZE, random_state=RANDOM_STATE, shuffle=True
)

train_ids = set(train_ids)

val_ids = set(val_ids)


def link_or_copy_image(src, dst):

    if dst.exists() or dst.is_symlink():
        return

    try:
        os.symlink(src, dst)

    except OSError:
        shutil.copy2(src, dst)


def process_one_image(image_id, group, images_dir, labels_dir):

    image_path = IMAGES_DIR / f"{image_id}.jpg"

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found for ID {image_id}: {image_path}")

    target_image_path = images_dir / image_path.name

    target_label_path = labels_dir / f"{image_id}.txt"

    link_or_copy_image(image_path, target_image_path)

    label_lines = []

    image_width = float(group["image_width"].iloc[0])

    image_height = float(group["image_height"].iloc[0])

    for _, row in group.iterrows():
        class_id = CLASS_TO_ID[row["class"]]

        x, y, w, h = parse_bbox(row["bbox"])

        polygon_pts = parse_points(row.get("points"))
        if len(polygon_pts) < 3:
            polygon_pts = bbox_to_polygon(x, y, w, h)

        norm_poly = normalize_polygon(polygon_pts, image_width, image_height)
        if len(norm_poly) < 3:
            continue

        coords = " ".join(f"{px:.6f} {py:.6f}" for px, py in norm_poly)
        label_lines.append(f"{class_id} {coords}")

    target_label_path.write_text("\n".join(label_lines) + "\n", encoding="utf-8")

    return image_id


def write_labels(
    split_name, split_ids, images_dir, labels_dir, n_jobs=N_JOBS_DATA_PREP
):

    split_rows = train_df[train_df["Image_ID"].isin(split_ids)].copy()

    grouped = list(split_rows.groupby("Image_ID", sort=False))

    processed = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(process_one_image)(image_id, group, images_dir, labels_dir)
        for image_id, group in grouped
    )

    print(f"{split_name}: prepared {len(processed)} images")


write_labels("train", train_ids, YOLO_IMAGES_TRAIN, YOLO_LABELS_TRAIN)

write_labels("val", val_ids, YOLO_IMAGES_VAL, YOLO_LABELS_VAL)


data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": str(YOLO_IMAGES_TRAIN.relative_to(YOLO_DATASET_DIR)),
    "val": str(YOLO_IMAGES_VAL.relative_to(YOLO_DATASET_DIR)),
    "names": CLASS_NAMES,
}


DATA_YAML_PATH.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")


print("YOLO segmentation dataset prepared")

print(f"Train IDs: {len(train_ids)}")

print(f"Val IDs:   {len(val_ids)}")

print(f"Data YAML:  {DATA_YAML_PATH}")


train: prepared 788 images
val: prepared 197 images
YOLO segmentation dataset prepared
Train IDs: 788
Val IDs:   197
Data YAML:  /content/yolo_dataset/data.yaml


# Part 3: Model Training



Train the YOLO model for 10 epochs with image size 640 and batch size 8.

In [9]:
model = YOLO(MODEL_WEIGHTS)


train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_DIR),
    name="yolo_train",
    exist_ok=True,
)


best_weights_path = Path(model.trainer.best)

print(f"Training complete. Best weights: {best_weights_path}")


Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

# Part 4: Inference



Run inference on test images using the best trained weights and collect per-detection predictions.

In [10]:
inference_model = YOLO(str(best_weights_path))


test_ids = test_df["Image_ID"].drop_duplicates().tolist()

predictions = []


for image_id in test_ids:
    image_path = IMAGES_DIR / f"{image_id}.jpg"

    if not image_path.exists():
        print(f"Skipping missing image: {image_path}")

        continue

    results = inference_model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
    )

    result = results[0]

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        continue

    for box in boxes:
        xyxy = box.xyxy[0].tolist()

        x1, y1, x2, y2 = xyxy

        xmin = max(0.0, x1)

        ymin = max(0.0, y1)

        xmax = max(0.0, x2)

        ymax = max(0.0, y2)

        cls_id = int(box.cls.item())

        confidence = float(box.conf.item())

        predictions.append(
            {
                "Image_ID": image_id,
                "class": CLASS_NAMES[cls_id],
                "confidence": confidence,
                "ymin": ymin,
                "xmin": xmin,
                "ymax": ymax,
                "xmax": xmax,
            }
        )


predictions_df = pd.DataFrame(predictions)

print(f"Predictions generated: {len(predictions_df)}")

predictions_df.head()


Predictions generated: 1572


,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,04DZeTP47eXR,Flower_open,0.860413,2056.944824,2362.783447,2140.260254,2473.107910
1,04DZeTP47eXR,Flower_open,0.805834,2752.086670,2372.939209,2895.545898,2473.810791
2,04DZeTP47eXR,Flower_open,0.696971,2233.296387,2172.430664,2333.789795,2244.548584
3,04DZeTP47eXR,Flower_open,0.549597,2144.375000,1713.942871,2199.149414,1793.847534
4,04DZeTP47eXR,Flower_open,0.492095,322.907867,1943.085083,424.959717,2023.197144


# Part 5: Submission Creation


Create the final submission file with Zindi columns `Image_ID`, `class`, `confidence`, `ymin`, `xmin`, `ymax`, and `xmax`.

In [11]:
submission_columns = ["Image_ID", "class", "confidence", "ymin", "xmin", "ymax", "xmax"]


if predictions_df.empty:
    submission_df = pd.DataFrame(columns=submission_columns)

else:
    submission_df = predictions_df[submission_columns].copy()

    submission_df = submission_df.sort_values(
        ["Image_ID", "confidence"], ascending=[True, False]
    ).reset_index(drop=True)


# Ensure every Test.csv Image_ID exists in submission at least once.
# Use a no-op placeholder class so missing-image rows are ignored by class-wise metrics.
if "test_df" not in globals():
    test_df = pd.read_csv(TEST_CSV)

test_ids = set(test_df["Image_ID"].astype(str).unique())
submission_ids = set(submission_df["Image_ID"].astype(str).unique()) if not submission_df.empty else set()
missing_ids = sorted(test_ids - submission_ids)

if missing_ids:
    placeholder_df = pd.DataFrame(
        {
            "Image_ID": missing_ids,
            "class": ["__background__"] * len(missing_ids),
            "confidence": [0.0] * len(missing_ids),
            "ymin": [0] * len(missing_ids),
            "xmin": [0] * len(missing_ids),
            "ymax": [0] * len(missing_ids),
            "xmax": [0] * len(missing_ids),
        }
    )
    submission_df = pd.concat([submission_df, placeholder_df], ignore_index=True)
    print(f"Added {len(missing_ids)} placeholder row(s) for missing Image_ID(s) in submission.")


# Force bbox coordinates to integer values
coord_cols = ["xmin", "ymin", "xmax", "ymax"]
for c in coord_cols:
    submission_df[c] = pd.to_numeric(submission_df[c], errors="coerce")

invalid = int(submission_df[coord_cols].isna().any(axis=1).sum())
if invalid > 0:
    raise ValueError(
        f"submission has {invalid} row(s) with invalid bbox coords; fix before integer casting"
    )

submission_df[coord_cols] = submission_df[coord_cols].round().astype(int)

# Keep consistent ordering after placeholder merge
submission_df = submission_df.sort_values(["Image_ID", "confidence"], ascending=[True, False]).reset_index(drop=True)


submission_df.to_csv(SUBMISSION_PATH, index=False)


print(f"Submission saved to: {SUBMISSION_PATH}")

print(f"Submission rows: {len(submission_df)}")
print(f"Unique submission Image_IDs: {submission_df['Image_ID'].nunique()} | Unique test Image_IDs: {len(test_ids)}")

submission_df.head()


Added 2 placeholder row(s) for missing Image_ID(s) in submission.
Submission saved to: /content/submission.csv
Submission rows: 1574
Unique submission Image_IDs: 352 | Unique test Image_IDs: 352


,Image_ID,class,confidence,ymin,xmin,ymax,xmax
0,04DZeTP47eXR,Flower_open,0.860413,2057,2363,2140,2473
1,04DZeTP47eXR,Flower_open,0.805834,2752,2373,2896,2474
2,04DZeTP47eXR,Flower_open,0.696971,2233,2172,2334,2245
3,04DZeTP47eXR,Flower_open,0.549597,2144,1714,2199,1794
4,04DZeTP47eXR,Flower_open,0.492095,323,1943,425,2023
